# 01 — Exploração das avaliações da Olist

Objetivo: entender os dados antes de modelar. Três perguntas guiam o notebook:

1. Quantas avaliações têm comentário (o nosso universo de trabalho)?
2. Como as notas se distribuem — e quem comenta mais, cliente satisfeito ou insatisfeito?
3. Qual o tamanho dos textos (impacta TF-IDF, embeddings e custo do LLM)?

A limpeza **não** é reimplementada aqui: usamos `reviews_triage.data`, o mesmo código que
gera `data/processed/`. Assim o que exploramos é exatamente o que os modelos vão ver.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from reviews_triage.data import clean, load_raw

# Funciona abrindo o notebook tanto da raiz do repo quanto de notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
raw = load_raw(ROOT / "data/raw/olist_order_reviews_dataset.csv")
df = clean(raw)
raw.shape, df.shape

## 1. Quantas avaliações têm comentário?

In [ ]:
raw_has_message = raw["review_comment_message"].fillna("").str.strip() != ""
summary = pd.Series(
    {
        "linhas no CSV": len(raw),
        "review_id únicos": raw["review_id"].nunique(),
        "com comentário (bruto)": int(raw_has_message.sum()),
        "com comentário, limpas e sem duplicata": len(df),
    }
)
summary.to_frame("n")

## 2. Distribuição das notas

Comparamos a distribuição de notas no dataset inteiro com a das avaliações que têm comentário.
Se forem diferentes, o comentário **não** é uma amostra neutra dos clientes.

In [ ]:
dist = pd.DataFrame(
    {
        "todas": raw["review_score"].value_counts(normalize=True),
        "com comentário": df["review_score"].value_counts(normalize=True),
    }
).sort_index()

comment_rate = raw.assign(has_message=raw_has_message).groupby("review_score")["has_message"].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
dist.plot.bar(ax=axes[0], rot=0, title="Distribuição das notas")
axes[0].set_xlabel("nota")
axes[0].set_ylabel("proporção")
comment_rate.plot.bar(ax=axes[1], rot=0, color="gray", title="% que deixa comentário, por nota")
axes[1].set_xlabel("nota")
axes[1].set_ylabel("taxa de comentário")
plt.tight_layout()
dist.round(3)

## 3. Tamanho dos textos

Medimos em caracteres e em palavras. Os percentis altos importam mais do que a média:
eles dizem se algum texto vai estourar o limite de tokens do modelo de embeddings.

In [ ]:
lengths = pd.DataFrame(
    {
        "chars": df["text"].str.len(),
        "words": df["text"].str.split().str.len(),
        "review_score": df["review_score"],
    }
)
lengths[["chars", "words"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
clip = lengths["words"].quantile(0.99)
lengths["words"].clip(upper=clip).plot.hist(bins=40, ax=axes[0])
axes[0].set_title("Palavras por avaliação (cortado no p99)")
axes[0].set_xlabel("palavras")
lengths.boxplot(column="words", by="review_score", ax=axes[1], showfliers=False)
axes[1].set_title("Palavras por nota (sem outliers)")
axes[1].set_xlabel("nota")
plt.suptitle("")
plt.tight_layout()

## 4. Exemplos por nota

Ler texto de verdade é a melhor forma de calibrar as categorias da Fase 2.

In [ ]:
pd.set_option("display.max_colwidth", 200)
df.groupby("review_score").sample(n=3, random_state=42)[["review_score", "text"]]

## 5. Textos repetidos

Avaliações idênticas ("Muito bom", "Recomendo") são reais e ficam no dataset, mas no gold set
(Fase 2) seria desperdício rotular o mesmo texto duas vezes.

In [ ]:
df["text"].str.lower().value_counts().head(10).to_frame("ocorrências")

## 6. Resumo em números

In [ ]:
words = lengths["words"]
low_commented = (df["review_score"] <= 2).mean()
low_all = (raw["review_score"] <= 2).mean()
repeated = df["text"].str.lower().duplicated().mean()
p50, p90, p99 = words.quantile([0.5, 0.9, 0.99])

print(f"Com comentário: {len(df):,} de {len(raw):,} linhas ({len(df) / len(raw):.1%})")
print(f"Notas 1–2: {low_commented:.1%} das comentadas vs. {low_all:.1%} do dataset todo")
print(f"Palavras por avaliação: mediana {p50:.0f}, p90 {p90:.0f}, p99 {p99:.0f}")
print(f"Textos com mais de 100 palavras: {(words > 100).mean():.2%}")
print(f"Textos repetidos (mesma string, ignorando caixa): {repeated:.1%}")

## O que isso muda nas próximas fases

- **Desbalanceamento.** Se as notas são concentradas em poucos valores, as categorias também
  devem ser. Por isso a métrica principal será **F1 macro** (cada categoria pesa igual), e não
  accuracy, que premia acertar só a classe majoritária.
- **Gold set estratificado.** Uma amostra puramente aleatória de 80 avaliações pode trazer poucos
  exemplos de problemas raros. Na Fase 2 vamos estratificar por nota e evitar textos repetidos.
- **Textos curtos.** Se a mediana fica em poucas dezenas de palavras, o TF-IDF tem pouco sinal
  por documento e o modelo de embeddings (limite de 128 tokens no MiniLM multilíngue) quase
  nunca trunca nada.
- **Custo.** Textos curtos significam que o custo por avaliação vai ser dominado pelo prompt e
  pela política recuperada, não pela avaliação em si. Vamos medir isso na Fase 5.